### Create patient dataset (per visit/HADM_ID)

1. select patients with less than 5 diseases from DIAGNOSES_ICD.csv

In [48]:
import pandas as pd
import numpy as np


In [49]:
# filter DIAGNOSES_ICD.csv for SUBJECT_ID and HADM_ID appearing less than 5 times
diagnoses = pd.read_csv('mimic-iii-clinical-database-1.4\DIAGNOSES_ICD.csv')
# Filter SUBJECT_IDs that appear less than 5 times
subject_counts = diagnoses['SUBJECT_ID'].value_counts()
subjects_to_keep = subject_counts[subject_counts < 5].index
diagnoses = diagnoses[diagnoses['SUBJECT_ID'].isin(subjects_to_keep)]

# Filter HADM_IDs that appear less than 5 times
hadm_counts = diagnoses['HADM_ID'].value_counts()
hadms_to_keep = hadm_counts[hadm_counts < 5].index
diagnoses = diagnoses[diagnoses['HADM_ID'].isin(hadms_to_keep)]

diagnoses

,ROW_ID,SUBJECT_ID,HADM_ID,SEQ_NUM,ICD9_CODE
0,1,2,163353,1.0,V3001
50,100041,8958,168335,1.0,85306
51,100042,8958,168335,2.0,8730
52,100043,8958,168335,3.0,9130
53,100044,8958,168335,4.0,E8150
...,...,...,...,...,...
651023,99978,8949,175113,2.0,5849
651024,99979,8949,175113,3.0,4019
651026,99980,8949,175113,4.0,34690
651027,99981,8950,120328,1.0,V3000


In [50]:
# map ICD9 codes with 'mimic-iii-clinical-database-1.4\D_ICD_DIAGNOSES.csv'
icd = pd.read_csv('mimic-iii-clinical-database-1.4\D_ICD_DIAGNOSES.csv')
icd = icd[['ICD9_CODE', 'SHORT_TITLE', 'LONG_TITLE']]
diagnoses_merged = diagnoses.merge(icd, on='ICD9_CODE', how='left')
diagnoses_merged

,ROW_ID,SUBJECT_ID,HADM_ID,SEQ_NUM,ICD9_CODE,SHORT_TITLE,LONG_TITLE
0,1,2,163353,1.0,V3001,Single lb in-hosp w cs,"Single liveborn, born in hospital, delivered b..."
1,100041,8958,168335,1.0,85306,Brain hem NEC-coma NOS,Other and unspecified intracranial hemorrhage ...
2,100042,8958,168335,2.0,8730,Open wound of scalp,"Open wound of scalp, without mention of compli..."
3,100043,8958,168335,3.0,9130,Abrasion forearm,"Abrasion or friction burn of elbow, forearm, a..."
4,100044,8958,168335,4.0,E8150,Mv coll w oth obj-driver,Other motor vehicle traffic accident involving...
...,...,...,...,...,...,...,...
20693,99978,8949,175113,2.0,5849,Acute kidney failure NOS,"Acute kidney failure, unspecified"
20694,99979,8949,175113,3.0,4019,Hypertension NOS,Unspecified essential hypertension
20695,99980,8949,175113,4.0,34690,Migrne unsp wo ntrc mgrn,"Migraine, unspecified, without mention of intr..."
20696,99981,8950,120328,1.0,V3000,Single lb in-hosp w/o cs,"Single liveborn, born in hospital, delivered w..."


In [51]:
# set of HADM_IDs
hadm_ids = set(diagnoses_merged['HADM_ID'].unique())
len(hadm_ids)

6446

2. For each HADM_ID, check whether:
   - Not newborn (`ADMISSION_TYPE != 'NEWBORN'` in `ADMISSIONS.csv`)
   - Not dead (`HOSPITAL_EXPIRE_FLAG != 1`)
   - Has chart events data (`HAS_CHARTEVENTS_DATA != 0`)

In [52]:
# Step 1: Load ADMISSIONS.csv
adm = pd.read_csv('mimic-iii-clinical-database-1.4/ADMISSIONS.csv')

# Step 2: Filter for HADM_IDs in your diagnosis set
adm_filtered = adm[adm['HADM_ID'].isin(hadm_ids)]
 
adm_filtered = adm_filtered[
    (adm_filtered['ADMISSION_TYPE'] != 'NEWBORN') &
    (adm_filtered['HOSPITAL_EXPIRE_FLAG'] != 1)&
    (adm_filtered['HAS_CHARTEVENTS_DATA'] != 0)
]

adm_filtered

,ROW_ID,SUBJECT_ID,HADM_ID,ADMITTIME,DISCHTIME,DEATHTIME,ADMISSION_TYPE,ADMISSION_LOCATION,DISCHARGE_LOCATION,INSURANCE,LANGUAGE,RELIGION,MARITAL_STATUS,ETHNICITY,EDREGTIME,EDOUTTIME,DIAGNOSIS,HOSPITAL_EXPIRE_FLAG,HAS_CHARTEVENTS_DATA
3,24,24,161859,2139-06-06 16:14:00,2139-06-09 12:48:00,NaN,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,HOME,Private,NaN,PROTESTANT QUAKER,SINGLE,WHITE,NaN,NaN,INTERIOR MYOCARDIAL INFARCTION,0,1
4,25,25,129635,2160-11-02 02:06:00,2160-11-05 14:55:00,NaN,EMERGENCY,EMERGENCY ROOM ADMIT,HOME,Private,NaN,UNOBTAINABLE,MARRIED,WHITE,2160-11-02 01:01:00,2160-11-02 04:27:00,ACUTE CORONARY SYNDROME,0,1
11,32,33,176176,2116-12-23 22:30:00,2116-12-27 12:05:00,NaN,EMERGENCY,EMERGENCY ROOM ADMIT,HOME,Medicare,NaN,PROTESTANT QUAKER,MARRIED,UNKNOWN/NOT SPECIFIED,2116-12-23 14:42:00,2116-12-23 23:41:00,SEPSIS;TELEMETRY,0,1
36,470,365,139873,2147-12-15 17:10:00,2147-12-23 12:17:00,NaN,EMERGENCY,CLINIC REFERRAL/PREMATURE,HOME,Private,NaN,JEWISH,MARRIED,WHITE,NaN,NaN,CORONARY ARTERY DISEASE\CARDIAC CATH,0,1
46,44,43,146828,2186-10-01 23:15:00,2186-10-11 18:00:00,NaN,EMERGENCY,EMERGENCY ROOM ADMIT,REHAB/DISTINCT PART HOSP,Private,ENGL,NOT SPECIFIED,MARRIED,WHITE,2186-10-01 19:42:00,2186-10-01 23:46:00,T12 FRACTURE,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58838,57767,96249,132101,2107-09-10 07:51:00,2107-09-10 19:44:00,NaN,EMERGENCY,CLINIC REFERRAL/PREMATURE,HOME,Private,ENGL,UNOBTAINABLE,MARRIED,WHITE,2107-09-10 02:53:00,2107-09-10 08:53:00,BRAIN ARTERIAL CLOT,0,1
58876,56888,93556,154802,2121-03-31 07:15:00,2121-04-04 13:30:00,NaN,ELECTIVE,PHYS REFERRAL/NORMAL DELI,HOME HEALTH CARE,Medicare,ENGL,CATHOLIC,MARRIED,WHITE,NaN,NaN,CORONARY ARTERY DISEASE\CORONARY ARTERY BYPASS...,0,1
58907,57500,95408,121497,2150-07-14 17:36:00,2150-07-21 14:02:00,NaN,ELECTIVE,PHYS REFERRAL/NORMAL DELI,HOME HEALTH CARE,Medicare,ENGL,CATHOLIC,MARRIED,WHITE,NaN,NaN,THORACOABDOMINAL ANEURYSM/SDA,0,1
58908,57501,95409,139421,2159-01-30 01:33:00,2159-01-30 13:00:00,NaN,EMERGENCY,EMERGENCY ROOM ADMIT,HOME,Self Pay,ENGL,OTHER,SINGLE,WHITE,2159-01-29 23:11:00,2159-01-30 02:36:00,INTERACRANIAL HEMORRHAGE,0,1


In [53]:
duplicated_subjects = adm_filtered['SUBJECT_ID'].value_counts()
duplicated_subjects = duplicated_subjects[duplicated_subjects > 1].index
adm_filtered_unique = adm_filtered[~adm_filtered['SUBJECT_ID'].isin(duplicated_subjects)]
adm_filtered_unique

,ROW_ID,SUBJECT_ID,HADM_ID,ADMITTIME,DISCHTIME,DEATHTIME,ADMISSION_TYPE,ADMISSION_LOCATION,DISCHARGE_LOCATION,INSURANCE,LANGUAGE,RELIGION,MARITAL_STATUS,ETHNICITY,EDREGTIME,EDOUTTIME,DIAGNOSIS,HOSPITAL_EXPIRE_FLAG,HAS_CHARTEVENTS_DATA
3,24,24,161859,2139-06-06 16:14:00,2139-06-09 12:48:00,NaN,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,HOME,Private,NaN,PROTESTANT QUAKER,SINGLE,WHITE,NaN,NaN,INTERIOR MYOCARDIAL INFARCTION,0,1
4,25,25,129635,2160-11-02 02:06:00,2160-11-05 14:55:00,NaN,EMERGENCY,EMERGENCY ROOM ADMIT,HOME,Private,NaN,UNOBTAINABLE,MARRIED,WHITE,2160-11-02 01:01:00,2160-11-02 04:27:00,ACUTE CORONARY SYNDROME,0,1
11,32,33,176176,2116-12-23 22:30:00,2116-12-27 12:05:00,NaN,EMERGENCY,EMERGENCY ROOM ADMIT,HOME,Medicare,NaN,PROTESTANT QUAKER,MARRIED,UNKNOWN/NOT SPECIFIED,2116-12-23 14:42:00,2116-12-23 23:41:00,SEPSIS;TELEMETRY,0,1
36,470,365,139873,2147-12-15 17:10:00,2147-12-23 12:17:00,NaN,EMERGENCY,CLINIC REFERRAL/PREMATURE,HOME,Private,NaN,JEWISH,MARRIED,WHITE,NaN,NaN,CORONARY ARTERY DISEASE\CARDIAC CATH,0,1
46,44,43,146828,2186-10-01 23:15:00,2186-10-11 18:00:00,NaN,EMERGENCY,EMERGENCY ROOM ADMIT,REHAB/DISTINCT PART HOSP,Private,ENGL,NOT SPECIFIED,MARRIED,WHITE,2186-10-01 19:42:00,2186-10-01 23:46:00,T12 FRACTURE,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58838,57767,96249,132101,2107-09-10 07:51:00,2107-09-10 19:44:00,NaN,EMERGENCY,CLINIC REFERRAL/PREMATURE,HOME,Private,ENGL,UNOBTAINABLE,MARRIED,WHITE,2107-09-10 02:53:00,2107-09-10 08:53:00,BRAIN ARTERIAL CLOT,0,1
58876,56888,93556,154802,2121-03-31 07:15:00,2121-04-04 13:30:00,NaN,ELECTIVE,PHYS REFERRAL/NORMAL DELI,HOME HEALTH CARE,Medicare,ENGL,CATHOLIC,MARRIED,WHITE,NaN,NaN,CORONARY ARTERY DISEASE\CORONARY ARTERY BYPASS...,0,1
58907,57500,95408,121497,2150-07-14 17:36:00,2150-07-21 14:02:00,NaN,ELECTIVE,PHYS REFERRAL/NORMAL DELI,HOME HEALTH CARE,Medicare,ENGL,CATHOLIC,MARRIED,WHITE,NaN,NaN,THORACOABDOMINAL ANEURYSM/SDA,0,1
58908,57501,95409,139421,2159-01-30 01:33:00,2159-01-30 13:00:00,NaN,EMERGENCY,EMERGENCY ROOM ADMIT,HOME,Self Pay,ENGL,OTHER,SINGLE,WHITE,2159-01-29 23:11:00,2159-01-30 02:36:00,INTERACRANIAL HEMORRHAGE,0,1


In [54]:
hadm_ids = set(adm_filtered_unique['HADM_ID'].unique())
len(hadm_ids)

2819

3. for each HADM_ID, check whether 
   - has discharge summary
   - `ISERROR == 1` in NOTEEVENTS.csv

In [58]:
notes = pd.read_csv('mimic-iii-clinical-database-1.4/NOTEEVENTS.csv', low_memory=False)

In [59]:
# set HDAM_ID to integer
notes['HADM_ID'] = notes['HADM_ID'].astype('Int64')
filter_notes = notes[notes['HADM_ID'].isin(hadm_ids)]
filter_notes = filter_notes[
    (notes['CATEGORY'] == 'Discharge summary') &
    (notes['DESCRIPTION'] == 'Report') &
    (notes['ISERROR'] != 1)
]
filter_notes

C:\Users\Tianqi\AppData\Local\Temp\ipykernel_24724\3932834272.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  filter_notes = filter_notes[


,ROW_ID,SUBJECT_ID,HADM_ID,CHARTDATE,CHARTTIME,STORETIME,CATEGORY,DESCRIPTION,CGID,ISERROR,TEXT
75,222,93556,154802,2121-04-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2121-3-31**] ...
113,38,142,131357,2143-04-05,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2143-4-1**] Discharge ...
124,49,3445,196939,2141-07-10,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2141-7-4**] Discharg...
174,99,7008,147300,2131-10-22,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2131-10-16**] Discharg...
202,127,25486,153955,2174-09-02,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2174-8-31**] Dischar...
...,...,...,...,...,...,...,...,...,...,...,...
55853,53318,6513,143839,2199-07-15,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2199-7-13**] ...
55879,53344,41264,148124,2194-06-02,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2194-5-29**] ...
55881,53346,57780,175214,2107-03-12,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2107-3-10**] ...
55883,53348,18382,138924,2187-10-03,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2187-10-2**] Discharge...


In [ ]:
# remove row if SUBJECT_ID is duplicated
duplicated_subjects = filter_notes['SUBJECT_ID'].value_counts()
duplicated_subjects = duplicated_subjects[duplicated_subjects > 1].index
filter_notes_unique = filter_notes[~filter_notes['SUBJECT_ID'].isin(duplicated_subjects)]


Index([ 1817,  3473, 23428, 12955, 24398,  4371, 12488,   595,  3115, 15161,
       14904,  9989, 14149, 20563, 23030, 17168, 22049, 41405,  8545,   525,
        4623, 15808, 71458,  8216,  1993,  8429, 22563, 47243,   144, 22525,
       17098, 18598, 65541, 26062,  8234,  5648,   296, 23922, 16548, 21610,
        2373, 21361,  4681, 13776, 13567, 14806, 10685, 15719,  6047, 15783,
         217,  9769, 12696, 23891,  3989, 19886, 18176,  3863, 22422,  1783,
        8033, 13435, 12254,  4514,  9958, 17043, 24308],
      dtype='int64', name='SUBJECT_ID')

In [65]:
# remove row if SUBJECT_ID is duplicated
duplicated_subjects = filter_notes['SUBJECT_ID'].value_counts()
duplicated_subjects = duplicated_subjects[duplicated_subjects > 1].index
filter_notes_unique = filter_notes[~filter_notes['SUBJECT_ID'].isin(duplicated_subjects)]
filter_notes_unique

,ROW_ID,SUBJECT_ID,HADM_ID,CHARTDATE,CHARTTIME,STORETIME,CATEGORY,DESCRIPTION,CGID,ISERROR,TEXT
75,222,93556,154802,2121-04-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2121-3-31**] ...
113,38,142,131357,2143-04-05,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2143-4-1**] Discharge ...
124,49,3445,196939,2141-07-10,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2141-7-4**] Discharg...
174,99,7008,147300,2131-10-22,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2131-10-16**] Discharg...
202,127,25486,153955,2174-09-02,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2174-8-31**] Dischar...
...,...,...,...,...,...,...,...,...,...,...,...
55853,53318,6513,143839,2199-07-15,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2199-7-13**] ...
55879,53344,41264,148124,2194-06-02,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2194-5-29**] ...
55881,53346,57780,175214,2107-03-12,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2107-3-10**] ...
55883,53348,18382,138924,2187-10-03,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2187-10-2**] Discharge...


In [67]:
hadm_ids = set(filter_notes_unique['HADM_ID'].unique())
len(hadm_ids)

2452

In [68]:
# save hadm_ids to txt
with open('data/valid_hadm_ids.txt', 'w') as f:
    for hadm_id in hadm_ids:
        f.write(f"{hadm_id}\n")

In [69]:
# randomly sample 500 HADM_IDs from the set of valid HADM_IDs
import random
random.seed(42)
random_hadm_ids = random.sample(hadm_ids, 500)

# save random_hadm_ids to txt
with open('data/hadm_ids.txt', 'w') as f:
    for hadm_id in random_hadm_ids:
        f.write(f"{hadm_id}\n")

some merging and data cleaning

In [79]:
def filter_hadm_ids(input_file, output_file):
    """
    Filters the input CSV file for rows with HADM_IDs in the provided set and saves to output CSV.
    Overwrites the output file if it exists.
    """
    hadm = pd.read_csv('data/hadm_ids.txt', header=None)[0].astype(int)
    hadm_ids = set(hadm)

    chunk_size = 100000
    first_chunk = True

    with pd.read_csv(input_file, chunksize=chunk_size, low_memory=False) as reader:
        for i, chunk in enumerate(reader):
            filtered_chunk = chunk[chunk['HADM_ID'].isin(hadm_ids)]

            if not filtered_chunk.empty:
                mode = 'w' if first_chunk else 'a'  # overwrite only for first chunk
                filtered_chunk.to_csv(output_file, mode=mode, index=False, header=first_chunk)
                first_chunk = False

            print(f"Processed chunk {i + 1}, matched rows: {len(filtered_chunk)}")

    print("✅ Done!")


In [80]:
input_file = r'mimic-iii-clinical-database-1.4\ADMISSIONS.csv'
output_file = r'data/admissions.csv'
filter_hadm_ids(input_file, output_file)

Processed chunk 1, matched rows: 500
✅ Done!


In [81]:
input_file = r'mimic-iii-clinical-database-1.4\CHARTEVENTS.csv'
output_file = r'data/chartevents.csv'
filter_hadm_ids(input_file, output_file)

Processed chunk 1, matched rows: 0
Processed chunk 2, matched rows: 0
Processed chunk 3, matched rows: 0
Processed chunk 4, matched rows: 0
Processed chunk 5, matched rows: 0
Processed chunk 6, matched rows: 0
Processed chunk 7, matched rows: 0
Processed chunk 8, matched rows: 0
Processed chunk 9, matched rows: 0
Processed chunk 10, matched rows: 0
Processed chunk 11, matched rows: 0
Processed chunk 12, matched rows: 0
Processed chunk 13, matched rows: 0
Processed chunk 14, matched rows: 0
Processed chunk 15, matched rows: 0
Processed chunk 16, matched rows: 0
Processed chunk 17, matched rows: 0
Processed chunk 18, matched rows: 0
Processed chunk 19, matched rows: 0
Processed chunk 20, matched rows: 0
Processed chunk 21, matched rows: 0
Processed chunk 22, matched rows: 0
Processed chunk 23, matched rows: 0
Processed chunk 24, matched rows: 0
Processed chunk 25, matched rows: 0
Processed chunk 26, matched rows: 0
Processed chunk 27, matched rows: 0
Processed chunk 28, matched rows: 0
P

In [82]:
input_file = r'mimic-iii-clinical-database-1.4\DIAGNOSES_ICD.csv'
output_file = r'data/diagnoses.csv'
filter_hadm_ids(input_file, output_file)

Processed chunk 1, matched rows: 340
Processed chunk 2, matched rows: 357
Processed chunk 3, matched rows: 259
Processed chunk 4, matched rows: 163
Processed chunk 5, matched rows: 179
Processed chunk 6, matched rows: 179
Processed chunk 7, matched rows: 134
✅ Done!


In [83]:
input_file = r'mimic-iii-clinical-database-1.4\LABEVENTS.csv'
output_file = r'data/labevents.csv'
filter_hadm_ids(input_file, output_file)

Processed chunk 1, matched rows: 280
Processed chunk 2, matched rows: 344
Processed chunk 3, matched rows: 0
Processed chunk 4, matched rows: 361
Processed chunk 5, matched rows: 250
Processed chunk 6, matched rows: 529
Processed chunk 7, matched rows: 132
Processed chunk 8, matched rows: 913
Processed chunk 9, matched rows: 328
Processed chunk 10, matched rows: 320
Processed chunk 11, matched rows: 542
Processed chunk 12, matched rows: 831
Processed chunk 13, matched rows: 500
Processed chunk 14, matched rows: 275
Processed chunk 15, matched rows: 52
Processed chunk 16, matched rows: 440
Processed chunk 17, matched rows: 266
Processed chunk 18, matched rows: 180
Processed chunk 19, matched rows: 507
Processed chunk 20, matched rows: 84
Processed chunk 21, matched rows: 0
Processed chunk 22, matched rows: 170
Processed chunk 23, matched rows: 169
Processed chunk 24, matched rows: 158
Processed chunk 25, matched rows: 425
Processed chunk 26, matched rows: 222
Processed chunk 27, matched

In [84]:
input_file = r'mimic-iii-clinical-database-1.4\MICROBIOLOGYEVENTS.csv'
output_file = r'data/microbiologyevents.csv'
filter_hadm_ids(input_file, output_file)

Processed chunk 1, matched rows: 139
Processed chunk 2, matched rows: 232
Processed chunk 3, matched rows: 341
Processed chunk 4, matched rows: 161
Processed chunk 5, matched rows: 122
Processed chunk 6, matched rows: 166
Processed chunk 7, matched rows: 35
✅ Done!


In [85]:
input_file = r'mimic-iii-clinical-database-1.4\NOTEEVENTS.csv'
output_file = r'data/noteevents.csv'
filter_hadm_ids(input_file, output_file)

Processed chunk 1, matched rows: 681
Processed chunk 2, matched rows: 346
Processed chunk 3, matched rows: 334
Processed chunk 4, matched rows: 75
Processed chunk 5, matched rows: 319
Processed chunk 6, matched rows: 168
Processed chunk 7, matched rows: 241
Processed chunk 8, matched rows: 418
Processed chunk 9, matched rows: 395
Processed chunk 10, matched rows: 382
Processed chunk 11, matched rows: 278
Processed chunk 12, matched rows: 244
Processed chunk 13, matched rows: 304
Processed chunk 14, matched rows: 469
Processed chunk 15, matched rows: 510
Processed chunk 16, matched rows: 516
Processed chunk 17, matched rows: 223
Processed chunk 18, matched rows: 0
Processed chunk 19, matched rows: 14
Processed chunk 20, matched rows: 50
Processed chunk 21, matched rows: 24
✅ Done!


In [86]:
input_file = r'mimic-iii-clinical-database-1.4\PRESCRIPTIONS.csv'
output_file = r'data/prescriptions.csv'
filter_hadm_ids(input_file, output_file)

Processed chunk 1, matched rows: 659
Processed chunk 2, matched rows: 515
Processed chunk 3, matched rows: 337
Processed chunk 4, matched rows: 628
Processed chunk 5, matched rows: 772
Processed chunk 6, matched rows: 424
Processed chunk 7, matched rows: 417
Processed chunk 8, matched rows: 561
Processed chunk 9, matched rows: 568
Processed chunk 10, matched rows: 545
Processed chunk 11, matched rows: 897
Processed chunk 12, matched rows: 306
Processed chunk 13, matched rows: 816
Processed chunk 14, matched rows: 585
Processed chunk 15, matched rows: 520
Processed chunk 16, matched rows: 893
Processed chunk 17, matched rows: 441
Processed chunk 18, matched rows: 606
Processed chunk 19, matched rows: 735
Processed chunk 20, matched rows: 693
Processed chunk 21, matched rows: 336
Processed chunk 22, matched rows: 246
Processed chunk 23, matched rows: 397
Processed chunk 24, matched rows: 353
Processed chunk 25, matched rows: 502
Processed chunk 26, matched rows: 304
Processed chunk 27, m

In [87]:
input_file = r'mimic-iii-clinical-database-1.4\PROCEDURES_ICD.csv'
output_file = r'data/procedures.csv'
filter_hadm_ids(input_file, output_file)

Processed chunk 1, matched rows: 452
Processed chunk 2, matched rows: 529
Processed chunk 3, matched rows: 284
✅ Done!


In [88]:
# map ICD9 codes with 'mimic-iii-clinical-database-1.4\D_ICD_DIAGNOSES.csv'
icd = pd.read_csv('mimic-iii-clinical-database-1.4\D_ICD_DIAGNOSES.csv')
icd = icd[['ICD9_CODE', 'SHORT_TITLE', 'LONG_TITLE']]
diagnoses = pd.read_csv('data\diagnoses.csv')
diagnoses_merged = diagnoses.merge(icd, on='ICD9_CODE', how='left')
# Save the merged DataFrame to a CSV file
diagnoses_merged.to_csv('data/diagnoses.csv', index=False)
diagnoses_merged

,ROW_ID,SUBJECT_ID,HADM_ID,SEQ_NUM,ICD9_CODE,SHORT_TITLE,LONG_TITLE
0,102001,9116,152975,1.0,80702,Fracture two ribs-closed,Closed fracture of two ribs
1,102002,9116,152975,2.0,86509,Spleen injury NEC-closed,Other injury into spleen without mention of op...
2,102003,9116,152975,3.0,E8494,Accid in recreation area,Accidents occurring in place for recreation an...
3,102028,9120,100510,1.0,431,Intracerebral hemorrhage,Intracerebral hemorrhage
4,102029,9120,100510,2.0,2761,Hyposmolality,Hyposmolality and/or hyponatremia
...,...,...,...,...,...,...,...
1606,99525,8911,174845,2.0,4240,Mitral valve disorder,Mitral valve disorders
1607,99526,8911,174845,3.0,2720,Pure hypercholesterolem,Pure hypercholesterolemia
1608,99527,8911,174845,4.0,4019,Hypertension NOS,Unspecified essential hypertension
1609,9955,856,101226,1.0,430,Subarachnoid hemorrhage,Subarachnoid hemorrhage


In [89]:
icd = pd.read_csv('mimic-iii-clinical-database-1.4\D_ICD_DIAGNOSES.csv')
icd = icd[['ICD9_CODE', 'SHORT_TITLE', 'LONG_TITLE']]
procedures = pd.read_csv('data\procedures.csv')
icd['ICD9_CODE'] = icd['ICD9_CODE'].astype(str)
procedures['ICD9_CODE'] = procedures['ICD9_CODE'].astype(str)
procedures_merged = procedures.merge(icd, on='ICD9_CODE', how='left')
# Save the merged DataFrame to a CSV file
procedures_merged.to_csv('data/procedures.csv', index=False)
procedures_merged

,ROW_ID,SUBJECT_ID,HADM_ID,SEQ_NUM,ICD9_CODE,SHORT_TITLE,LONG_TITLE
0,100324,6310,144225,1,3612,Serous retina detachment,Serous retinal detachment
1,100325,6310,144225,2,3722,NaN,NaN
2,100326,6310,144225,3,3615,NaN,NaN
3,100327,6310,144225,4,3961,Mitral stenos/aort insuf,Mitral valve stenosis and aortic valve insuffi...
4,100328,6310,144225,5,8856,NaN,NaN
...,...,...,...,...,...,...,...
1260,98228,1800,140939,2,22,NaN,NaN
1261,98229,1800,140939,3,9672,Poisoning-paraldehyde,Poisoning by paraldehyde
1262,98230,1800,140939,4,3893,NaN,NaN
1263,98231,1800,140939,5,8872,"Amput abv elbow, unilat",Traumatic amputation of arm and hand (complete...


In [91]:
# map admissions with 'mimic-iii-clinical-database-1.4\PATIENTS.csv' on 'SUBJECT_ID'
patients = pd.read_csv('mimic-iii-clinical-database-1.4/PATIENTS.csv')
patients = patients[["SUBJECT_ID","GENDER","DOB","DOD","DOD_HOSP","DOD_SSN"]]
admissions = pd.read_csv('data/admissions.csv')
patients['SUBJECT_ID'] = patients['SUBJECT_ID'].astype(int)
admissions['SUBJECT_ID'] = admissions['SUBJECT_ID'].astype(int)
admissions = admissions.merge(patients, on='SUBJECT_ID', how='left')
# Save the merged DataFrame to a CSV file
admissions.to_csv('data/admissions.csv', index=False)
admissions

,ROW_ID,SUBJECT_ID,HADM_ID,ADMITTIME,DISCHTIME,DEATHTIME,ADMISSION_TYPE,ADMISSION_LOCATION,DISCHARGE_LOCATION,INSURANCE,...,EDREGTIME,EDOUTTIME,DIAGNOSIS,HOSPITAL_EXPIRE_FLAG,HAS_CHARTEVENTS_DATA,GENDER,DOB,DOD,DOD_HOSP,DOD_SSN
0,81,80,115385,2148-11-11 12:12:00,2148-11-16 11:53:00,NaN,EMERGENCY,EMERGENCY ROOM ADMIT,HOME,Self Pay,...,2148-11-11 11:03:00,2148-11-11 11:45:00,HEAD LACERATION,0,1,M,2104-12-13 00:00:00,NaN,NaN,NaN
1,82,81,175016,2192-01-09 18:50:00,2192-01-11 13:00:00,NaN,EMERGENCY,EMERGENCY ROOM ADMIT,HOSPICE-MEDICAL FACILITY,Medicare,...,2192-01-09 15:01:00,2192-01-09 19:30:00,INTRACRANIAL HEMORRHAGE;TELEMETRY,0,1,M,2106-12-20 00:00:00,2192-01-12 00:00:00,2192-01-12 00:00:00,2192-01-12 00:00:00
2,329,255,112013,2187-02-12 10:30:00,2187-02-15 10:30:00,NaN,ELECTIVE,PHYS REFERRAL/NORMAL DELI,HOME,Medicare,...,NaN,NaN,BENIGN PROSTATIC HYPERTROPHY/SDA,0,1,M,2109-08-05 00:00:00,NaN,NaN,NaN
3,341,265,101608,2149-02-20 07:15:00,2149-02-25 15:50:00,NaN,ELECTIVE,PHYS REFERRAL/NORMAL DELI,HOME HEALTH CARE,Private,...,NaN,NaN,CORONARY ARTERY DISEASE\CORONARY ARTERY BYPASS...,0,1,M,2093-08-01 00:00:00,NaN,NaN,NaN
4,1106,891,191029,2111-11-07 02:40:00,2111-11-10 12:48:00,NaN,EMERGENCY,EMERGENCY ROOM ADMIT,HOME,Private,...,2111-11-07 00:05:00,2111-11-07 03:13:00,BLEEDING LOW HCT,0,1,F,2074-05-27 00:00:00,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,56314,91685,147506,2142-12-17 07:15:00,2142-12-21 12:45:00,NaN,ELECTIVE,PHYS REFERRAL/NORMAL DELI,HOME HEALTH CARE,Private,...,NaN,NaN,CORONARY ARTERY DISEASE\CORONARY ARTERY BYPASS...,0,1,F,2093-06-01 00:00:00,NaN,NaN,NaN
496,57289,94798,194576,2164-10-17 08:00:00,2164-10-21 12:32:00,NaN,ELECTIVE,PHYS REFERRAL/NORMAL DELI,SNF,Medicare,...,NaN,NaN,CORONARY ARTERY DISEASE\CORONARY ARTERY BYPASS...,0,1,M,2095-03-11 00:00:00,NaN,NaN,NaN
497,57747,96197,187454,2178-07-26 21:02:00,2178-07-30 14:20:00,NaN,EMERGENCY,EMERGENCY ROOM ADMIT,HOME,Private,...,2178-07-26 18:50:00,2178-07-26 22:46:00,PULMONARY EMBOLIS,0,1,M,2128-08-05 00:00:00,NaN,NaN,NaN
498,56149,91222,127195,2104-07-20 18:09:00,2104-07-27 18:45:00,NaN,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,HOME HEALTH CARE,Private,...,NaN,NaN,CORONARY ARTERY DISEASE,0,1,M,2058-12-15 00:00:00,NaN,NaN,NaN


In [92]:
# map chartevents with 'mimic-iii-clinical-database-1.4\D_ITEMS.csv' on 'ITEMID'
chartevents = pd.read_csv('data/chartevents.csv')
items = pd.read_csv('mimic-iii-clinical-database-1.4/D_ITEMS.csv')
items = items[["ITEMID","LABEL","ABBREVIATION","DBSOURCE","LINKSTO","CATEGORY","UNITNAME","PARAM_TYPE","CONCEPTID"]]
chartevents['ITEMID'] = chartevents['ITEMID'].astype(int)
items['ITEMID'] = items['ITEMID'].astype(int)
chartevents = chartevents.merge(items, on='ITEMID', how='left')
# Save the merged DataFrame to a CSV file
chartevents.to_csv('data/chartevents.csv', index=False)
chartevents

C:\Users\Tianqi\AppData\Local\Temp\ipykernel_24724\1687717434.py:2: DtypeWarning: Columns (10,13,14) have mixed types. Specify dtype option on import or set low_memory=False.
  chartevents = pd.read_csv('data/chartevents.csv')


,ROW_ID,SUBJECT_ID,HADM_ID,ICUSTAY_ID,ITEMID,CHARTTIME,STORETIME,CGID,VALUE,VALUENUM,...,RESULTSTATUS,STOPPED,LABEL,ABBREVIATION,DBSOURCE,LINKSTO,CATEGORY,UNITNAME,PARAM_TYPE,CONCEPTID
0,3037137,25425,145566,221540.0,220179,2199-04-03 00:01:00,2199-04-03 00:02:00,15286.0,106.0,106.0,...,NaN,NaN,Non Invasive Blood Pressure systolic,NBPs,metavision,chartevents,Routine Vital Signs,mmHg,Numeric,NaN
1,3037138,25425,145566,221540.0,220180,2199-04-03 00:01:00,2199-04-03 00:02:00,15286.0,62.0,62.0,...,NaN,NaN,Non Invasive Blood Pressure diastolic,NBPd,metavision,chartevents,Routine Vital Signs,mmHg,Numeric,NaN
2,3037139,25425,145566,221540.0,220181,2199-04-03 00:01:00,2199-04-03 00:02:00,15286.0,73.0,73.0,...,NaN,NaN,Non Invasive Blood Pressure mean,NBPm,metavision,chartevents,Routine Vital Signs,mmHg,Numeric,NaN
3,3037140,25425,145566,221540.0,220045,2199-04-03 01:00:00,2199-04-03 01:04:00,15286.0,61.0,61.0,...,NaN,NaN,Heart Rate,HR,metavision,chartevents,Routine Vital Signs,bpm,Numeric,NaN
4,3037141,25425,145566,221540.0,220179,2199-04-03 01:00:00,2199-04-03 01:04:00,15286.0,82.0,82.0,...,NaN,NaN,Non Invasive Blood Pressure systolic,NBPs,metavision,chartevents,Routine Vital Signs,mmHg,Numeric,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1130913,329008001,98438,148925,275847.0,220048,2165-12-30 08:00:00,2165-12-30 08:17:00,17114.0,SR (Sinus Rhythm),NaN,...,NaN,NaN,Heart Rhythm,Heart Rhythm,metavision,chartevents,Routine Vital Signs,NaN,Text,NaN
1130914,329008002,98438,148925,275847.0,220739,2165-12-30 08:00:00,2165-12-30 07:49:00,17114.0,To Speech,3.0,...,NaN,NaN,GCS - Eye Opening,Eye Opening,metavision,chartevents,Neurological,NaN,Text,NaN
1130915,329008003,98438,148925,275847.0,223753,2165-12-30 08:00:00,2165-12-30 07:49:00,17114.0,Calm/Cooperative,4.0,...,NaN,NaN,Riker-SAS Scale,Riker-SAS Scale,metavision,chartevents,Pain/Sedation,NaN,Text,NaN
1130916,329008004,98438,148925,275847.0,223754,2165-12-30 08:00:00,2165-12-30 07:49:00,17114.0,No,NaN,...,NaN,NaN,Risk for Falls,Risk for Falls,metavision,chartevents,Neurological,NaN,Text,NaN


In [93]:
# map labevents with 'mimic-iii-clinical-database-1.4\D_LABITEMS.csv' on 'ITEMID'
labevents = pd.read_csv('data/labevents.csv')
items = pd.read_csv('mimic-iii-clinical-database-1.4/D_LABITEMS.csv')
items = items[["ITEMID","LABEL","FLUID","CATEGORY","LOINC_CODE"]]
labevents = labevents.merge(items, on='ITEMID', how='left')
# Save the merged DataFrame to a CSV file
labevents.to_csv('data/labevents.csv', index=False)
labevents

,ROW_ID,SUBJECT_ID,HADM_ID,ITEMID,CHARTTIME,VALUE,VALUENUM,VALUEUOM,FLAG,LABEL,FLUID,CATEGORY,LOINC_CODE
0,36320,80,115385.0,51274,2148-11-12 02:30:00,12.5,12.50,sec,NaN,PT,Blood,Hematology,5902-2
1,36321,80,115385.0,51275,2148-11-12 02:30:00,24.2,24.20,sec,NaN,PTT,Blood,Hematology,3173-2
2,36322,80,115385.0,51277,2148-11-12 02:30:00,12.1,12.10,%,NaN,RDW,Blood,Hematology,788-0
3,36323,80,115385.0,51279,2148-11-12 02:30:00,4.78,4.78,m/uL,NaN,Red Blood Cells,Blood,Hematology,789-8
4,36324,80,115385.0,51301,2148-11-12 02:30:00,14.4,14.40,K/uL,abnormal,White Blood Cells,Blood,Hematology,804-5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
88905,27387887,96197,187454.0,51274,2178-07-30 05:35:00,32.1,32.10,sec,abnormal,PT,Blood,Hematology,5902-2
88906,27387888,96197,187454.0,51275,2178-07-30 05:35:00,32.2,32.20,sec,NaN,PTT,Blood,Hematology,3173-2
88907,27387889,96197,187454.0,51277,2178-07-30 05:35:00,12.3,12.30,%,NaN,RDW,Blood,Hematology,788-0
88908,27387890,96197,187454.0,51279,2178-07-30 05:35:00,4.35,4.35,m/uL,abnormal,Red Blood Cells,Blood,Hematology,789-8


In [94]:
# Load microbiology events
micro = pd.read_csv('data/microbiologyevents.csv')

# Load item metadata from D_ITEMS
items = pd.read_csv('mimic-iii-clinical-database-1.4/D_ITEMS.csv')
items = items[["ITEMID", "LABEL", "CATEGORY", "UNITNAME", "PARAM_TYPE", "CONCEPTID"]]
items['ITEMID'] = items['ITEMID'].astype(int)

# Convert item ID columns to int (if not null)
micro['SPEC_ITEMID'] = micro['SPEC_ITEMID'].astype('Int64')
micro['ORG_ITEMID'] = micro['ORG_ITEMID'].astype('Int64')
micro['AB_ITEMID'] = micro['AB_ITEMID'].astype('Int64')

# Merge for SPEC_ITEMID
micro = micro.merge(items, left_on='SPEC_ITEMID', right_on='ITEMID', how='left', suffixes=('', '_SPEC'))

# Merge for ORG_ITEMID
micro = micro.merge(items, left_on='ORG_ITEMID', right_on='ITEMID', how='left', suffixes=('', '_ORG'))

# Merge for AB_ITEMID
micro = micro.merge(items, left_on='AB_ITEMID', right_on='ITEMID', how='left', suffixes=('', '_AB'))

# Optional: drop redundant ITEMID columns from merges
micro = micro.drop(columns=['ITEMID', 'ITEMID_ORG', 'ITEMID_AB'])
micro

,ROW_ID,SUBJECT_ID,HADM_ID,CHARTDATE,CHARTTIME,SPEC_ITEMID,SPEC_TYPE_DESC,ORG_ITEMID,ORG_NAME,ISOLATE_NUM,...,LABEL_ORG,CATEGORY_ORG,UNITNAME_ORG,PARAM_TYPE_ORG,CONCEPTID_ORG,LABEL_AB,CATEGORY_AB,UNITNAME_AB,PARAM_TYPE_AB,CONCEPTID_AB
0,571,80,115385,2148-11-11 00:00:00,2148-11-11 11:00:00,70079,URINE,<NA>,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,572,80,115385,2148-11-11 00:00:00,2148-11-11 13:02:00,70091,MRSA SCREEN,<NA>,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,573,80,115385,2148-11-11 00:00:00,2148-11-11 13:03:00,70069,SWAB,<NA>,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,574,81,175016,2192-01-09 00:00:00,2192-01-09 17:00:00,70079,URINE,<NA>,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,575,81,175016,2192-01-09 00:00:00,2192-01-09 21:39:00,70091,MRSA SCREEN,<NA>,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1191,625448,98438,148925,2165-12-30 00:00:00,2165-12-30 01:10:00,70079,URINE,<NA>,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1192,625449,98438,148925,2165-12-30 00:00:00,2165-12-30 03:06:00,70062,SPUTUM,80075,YEAST,1.0,...,YEAST,ORGANISM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1193,625450,98438,148925,2165-12-30 00:00:00,2165-12-30 06:08:00,70042,Influenza A/B by DFA,<NA>,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1194,625451,98438,148925,2165-12-30 00:00:00,NaN,70012,BLOOD CULTURE,<NA>,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [95]:
# Save result
micro.to_csv('data/microbiologyevents_merged.csv', index=False)

Process discharge JSONs

In [3]:
import os
import json
import re

# Paths
input_dir = "data/patient_json"
output_dir = "data/patient_json_filtered"
os.makedirs(output_dir, exist_ok=True)

# Keywords for filtering (case insensitive)
keep_keywords = [
    'history', 'medications', 'allergies', 'physical',
    'exam', 'examination', 'review', 'complaint',
    'surgical', 'invasion', 'symptom', 'symptoms', 'summary', 'diagnosis', 'diagnoses'
]
exclude_keyword = 'discharge'

# Rule 2: Recursive keyword match
def contains_keyword(key):
    return any(k in key.lower() for k in keep_keywords)


def filter_recursive(data):
    if isinstance(data, dict):
        new_dict = {}
        for key, value in data.items():
            key_lower = key.lower()

            # Always include key if it matches directly
            if contains_keyword(key_lower):
                new_dict[key] = value
            else:
                # Recurse into children
                filtered_value = filter_recursive(value)

                # Only keep key if child contains matches
                if filtered_value:
                    new_dict[key] = filtered_value

        return new_dict if new_dict else None

    elif isinstance(data, list):
        # Filter each list item
        filtered_list = [filter_recursive(item) for item in data]
        filtered_list = [item for item in filtered_list if item]
        return filtered_list if filtered_list else None

    else:
        return None  # Drop all primitive values unless wrapped in valid key


# Process each file
for filename in os.listdir(input_dir):
    if not filename.endswith(".json"):
        continue

    filepath = os.path.join(input_dir, filename)
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()

    # Rule 1: Check for >3 empty sections
    if content.count(":{}") > 3:
        print(f"⚠️  {filename} contains > 3 empty sections")

    try:
        data = json.loads(content)
    except json.JSONDecodeError as e:
        print(f"❌ Failed to parse {filename}: {e}")
        continue

    # Rule 3: Remove top-level sections containing 'discharge'
    data = {
        key: value for key, value in data.items()
        if exclude_keyword not in key.lower()
    }

    # Rule 2: Recursively keep only relevant sections
    filtered_data = filter_recursive(data)


    # Rule 4 (if needed): Warn if content length is very short
    filtered_str = json.dumps(filtered_data, ensure_ascii=False)
    if len(filtered_str.strip()) < 50:
        print(f"📄 {filename} has very short content (length < 10 chars)")

    # Save cleaned file
    output_path = os.path.join(output_dir, filename)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(filtered_data, f, indent=2, ensure_ascii=False)
